# FEVER — snapshot RAG evaluation on all 19,998 claims
 **Stop the old live-download evaluation. Open this notebook with T4 GPU and choose Runtime → Run all. Evaluation is enabled. No retraining.**

 The article source is now the Wikipedia text distributed with FEVER. Existing
 claim-only retrieval rankings select the same top five page IDs; no reference
 label or annotated evidence is used to select text. All available article text
 in those snapshot records is read, chunked and embedded into a temporary RAG
 database for each claim. This uses the dataset's supplied text, not the full
 current Wikipedia website with its later edits, tables and metadata.

 The source package contains all 24,319 selected pages, with none missing or
 empty. It is already uploaded to FEVER_Project. Article preparation makes zero
 Wikipedia HTTP calls. The unchanged MiniLM embedding model may download once
 (~0.08 GiB) if not cached. Static source data remain available; per-claim working
 articles, chunks and vector databases are deleted after use. Saved traces retain
 supplied passages, source-record hashes and exact character spans. The FEVER
 dump does not supply per-page revision IDs, so no revision IDs are invented.

 Both the original RoBERTa and the validation-selected last-3 model receive the
 exact same retrieved context and 512-token budget. The last-3 model's 69.42% was
 validation macro-F1, not final accuracy. Model weights and previous results are
 unchanged. This evidence-source change uses a new results folder and starts a
 separate comparison; it does not reuse live-Wikipedia predictions.

 A local 20-claim check verified evidence preparation and citation spans with no
 network calls. This notebook first runs a resumable 10-pair execution check, then
 continues toward all 19,998 within the session budget. These checks do not prove
 accuracy or completion speed. Actual Colab progress and ETA are measured during
 the run. Use Run all again after a pause to resume saved snapshot results.

## 1. Check GPU

In [1]:
import sys, subprocess
from pathlib import Path
import torch
print("Python:", sys.version.split()[0])
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then run this cell again.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv"], check=True)

Python: 3.13.16
GPU: Tesla T4
VRAM: 14.6 GiB


CompletedProcess(args=['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], returncode=0)

## 2. Install runtime libraries

In [2]:
import subprocess, sys, os
os.environ["USE_TF"] = "0"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HOME"] = "/content/fever_hf_cache"
packages = ["transformers==4.57.6", "accelerate>=1.10,<2", "bitsandbytes>=0.48,<0.49",
            "faiss-cpu==1.12.0", "scikit-learn>=1.6,<2", "requests>=2.32,<3", "librosa>=0.11,<0.12"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])
print("Libraries installed. If Colab requests a runtime restart, restart and rerun from section 1.")

Libraries installed. If Colab requests a runtime restart, restart and rerun from section 1.


## 3. Connect Drive and verify the original bundle
Your training notebook, model checkpoints and earlier predictions are preserved.

In [3]:
from google.colab import drive
from pathlib import Path
import zipfile, json, sys

drive.mount("/content/drive")
DRIVE_PROJECT = Path("/content/drive/MyDrive/FEVER_Project")
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
locations = [DRIVE_PROJECT / "FEVER_Colab_Bundle.zip",
             Path("/content/drive/MyDrive/FEVER_Colab_Bundle.zip")]
archive = next((p for p in locations if p.is_file()), None)
if archive is None:
    raise FileNotFoundError("The FEVER_Project folder is ready in My Drive. Upload FEVER_Colab_Bundle.zip into it, wait for the upload to finish, then rerun this cell. No path needs typing.")
print("Found your data:", archive)
PROJECT = Path("/content/fever_colab_project")
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    # Refuse traversal and links in an uploaded ZIP.
    for info in z.infolist():
        target = (PROJECT / info.filename).resolve()
        target.relative_to(PROJECT.resolve())
        if (info.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("Archive symlink rejected")
    z.extractall(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
import fever_colab_runtime as fever
EVAL_ROWS, EVAL_LABELS = fever.verify_bundle(PROJECT)
print("Project ready:", PROJECT)
# Install the debug runtime embedded in this notebook; the original Drive ZIP is unchanged.
import base64, zlib, hashlib, importlib
patch_source = zlib.decompress(base64.b64decode('eJy9fety20bW4H89BZypnQZsCKLsxBuTQaYcWc74G99WtjO7xbDwgSQoISIBDgBKVjyq2qfZB9sn2XPrC0CQUjyzX2rGIoC+9+lzP6e/+eabk3KZTr10nq6brPIWZeVln/O6yYtzb7FZLg/Tqslny8w7e/5z5L0tveuyuvSg1Ly8LpZlOq+9svDy1bqsmujg4F2ReT+//+Stynm29NLGS70mX2UjKLHaNOkUGlpn1eFsmeYrb3aRzS7XZV401MiLKr/KIu/jRV578L/UK7Lrg7++PPr2cJo3MCqoCE0VTegVJTZcZetleuOVC6+5yLwsrZY5zODdcpmuUu/7n7xVltabKsMq0cE333xzsKhK6LRcLrNZk5dFLcP2TspNAZPn7/O0yXDI+qN+DvGf38siO5AP57PwIq0vlvk05DfwK1plDSxlk4a/1WURrtLmIizrsEqLebkKK2gkW60X+ZJb4w7XUAiq6v7ew+PB6+c/nb7+EI/Vh0/v3787+/hBhers9OWnj6f46+27j97p23effv6r9+rty3dqcvDm3Qss/0X94zor1FD9D/hzhP88jr47fPLT4auibqrNrIHKtDxQBEd6SA9Hr/HfwyfR41bR24ODebbw6os0wRH7OM5geOBdxDLtCD49/u6pHxx413lzQSPnUlG5zgpfVVMVeGntLaAWQRZMEmAFltqHHqfzdLiIqiyd+8eDx98+xH+CcKpUMLyINmtcd38KbVdZs6kK7yK6yD7P8/OsbqBHGll+XqTwLfPL6W/BUJdrDw73IZpvVusaS4U1LHFymd3U8cdqk4VZgRCSpPUsz+OX6bLOgigrZgC8fhBsdYhjTaryupa1kB7H1AedBb8OaKI1TtRZD6rZZJ+hpaheL3OAlSKr/cDLYRoRLHi+9oMJdVKU1Spd5r9n0JbuQXkq+g3OiV9Hs7TOFuVyrhuCcUq1a9+U1zAb4VsNt9GmmQVRXpcL7EHP6bqC/UjKYsYbHF6ly02G2wwH5v1muszrCzgyq/UyazLvPz68ewsndXkDmCC7gsNGR3CWwWHUZebeusrmOR2wo1lZ4KQjPHwE57FdkxH+G63TCo/n6nKeVz4/6K1BLJSUl/QEYAArRTXoPSwdwhS8c9aePjsrHTyIaTbDKs3rzPsFf59WVVn56lSjOIuWEM8t0hkc+HyxyKp66KlHsDE8WOhf4PDAwyMcU18I9UmRrjJfReoRvcKnRyoC8J9D80ofDawjh+LaORMGOH0aaLjoA8jRIlosN/WFH4zKOlrUN8XMh1dwJIsSd9+Dt7IPPvYT0ohpcxdVlgEgIbxkVbiuyqYE9IdLx694O/g39EN/794L2rmYSx8p3WqEk1FfuU+6kZ6tei+f9L5E3llWbwA/88IC5i+r/Dwv0qVXZw1uah0ifZpdlGWNkPn29O9etSk83JoIdySDVR124d4sjkE4Fr3Yb7ioAPf54iaZboo5YMWqLBucI/7l5aQ3o1XszJjeHalVWuQLQCe8VC2sEBwQ3sAxhoxzEIOsxgr3uVaTCEa7sqtpkDI1jLVgEblezxL+RENlcgtL5y1SqDpHAKeaOPjrOrbYjUeLZOwoA8BM8mK9aWoa9VIFoypb1D2zs+WBoYA17p8lDB4A18d+YMjHz549+x53C999qcYqn6sJrUSF88dSt26xL3gipVjQKfcghv33cXBB74lfA9UH9JQul97xs5AaXCzyWQ6AMwdktizXyCl4r17UiscpbAE1GdEBrQlY9fsvy+FT+I+GscRhMNW+7ev+U5HpAZheaZkAcUJrusu0uPEVvVc0L5yzOoeT5jxmV/kcaFQmr1qL0DtzHHqK+NhbZATQtYeIOYU6vFNHaQEcFRWhgdAm4o7FXyw1glUnrm174QElQq9FPwDRp2SdL8vGABA0318YPuRzGke7xgEe7Toe3zWcz7f0+zP+HlPPSM8mE1pbbGI8mPyZ/h5PcC1xues/m+la0o1l+tbyre5/Liws0WCvBJywTNc40HUF7KyvfkEkkcNuM5oYapDL7GZQA/VI9gC6BJQLdHvk0dR5TQGbOYsCKLBGNqAhNCZoCtcwRBBldg2PZHKxSJryEo4U4oumuiH+C3nN87I8X2bRjHh+YTg3dVbh6hsSZ95E53Cg1F9fJh/f/e30LUHG51m2brxT+gMj0gwHkKCsuMorwAndOsw4rUrAtpo5hhO1LmHM/4ATBSciqQCka2gtfgtsClEnHOvF5hzQ+jnQ5Cy52Jjh/nXxfJ3zrPJiUcb0DNwQSRwJvtLNS6Pb/YSEVM1wmMbRgsWd5euZMhJvzVKcwdEF7kqA44SOkZfOZlldA3LFYQAv4L0EqCIWG+jSlI6dSEdc8KbcVHW2xMMP2Gk+9/TiISCydPYhm8FC1yR/pZumBPYtn3nLfAbsQkbtrJsUUELk0VC85madIXrHvz6wtAlxKUkS8MriMsORAjCOgRz4PmxZ2gBirUOFL1U4CPB0DBxe1seFjeocGEIgsPh1PBHutSJWBIkrMD01UmVgh+p0AcxgAew2YBRzLGhjlN4GFXKjF2morrP8/KIxQqWXroHofs5hotnyRoUVIMm5j4M7evzw4ZNB+DgI1c/5T94UmOEq81hErHg7vH9sUvjxO50Z56R8Udi/GnZGMbSjsKNO9EiS6U0DBHiIfd8SMAugI7Gg8yVw2ZTV7AK53NllCnQ4/rIebouFESAK7NRf89quCVMpqguyGZz5okb2HErBI27sMkOkAA8wwRoghIcTqgK4RlgYBYziZd4cLkH8hQVVQNrr+nC23qjJrTNxPSo11L9CdQ6FhtRzNNvM6bAnQAgBqJinHdD+OgXyOkmvgHVAdhlEF2SjCJRChZ91WzJDqsILRnxHgs8965UvSJrf2U3vSfuQoQivX3o/eicXaXGeGQBAuIe3H79FXUSITCLgyQyZQEDsmzWdI1RueCfvPxFvmAP+bUACyOaRg8Sh8lCFe5coVL+cPX/jASgONZT2lwdwXmcgYgAPMQiiBsjtEhDQqqxuBKKPA40rYS/rrGepzmeRKC985hZ2L5rzBSSC5iaZpbALUO3g7NPbj6/enCZnp7+8+vAKxLnYU0QODufZdHN+ePWdOjgA0gSICZb3TVnk0BaLgyfAHpSowKnK8wox1xrEEm++qUh+wunRiUPOpjwHwlaQiAi4Dilh7ciFdXRw8BEY9ynA4jktGmwRMgS88LQXsJsZkNQaSChJst6rBneJNT+MFS+zbH0IxPEqY/WOh8uXAAnImyTxEaWGIvkAuzPPseeQFj40wmo8ABxEBBZ+XcAZaqbAIiV1hjXq+PEgrC/K60TPmCnFBjqIhftghly2ieYA+BFeYe9Rn5DlfLiP1MsNmfHH5hc3JEAb0xLBBiySmTCngSlwniW6mFtHt70961X62V8A7mv8rW9BGA2+0023Vqb1pNtelrPL2CxLdPYanu3IynUCEFI0TonTK8KqXAIhi8AlHk/4zTStYiZe3H4NfSGzCNuHggGtAv6TFMipj2QM5wkJrM7KH/lKVwUhv90SUGzDdUovDQxvFbdaE2H+swpJXQRjj9WmWRx+71YDxB1/UbZpoCLtvgBz6v1UQwukiqAUESoCqzLQimXkZ6gEbhMpK48h9O7+p3AUG8D7CtAcspNIMhAm4A0BAr8iYKYRMkLXQyyy64T5VDUcQDFYCGSkNs1MDUnjFCo8DWqI/95q2dSBBDwemgMVHrT5x3wVIRdjzg28oM9mj/GNTxOLeQ3wTIPIFNv5z7N6Zk8Dn0n8J5zfAGrOZwlsy7I2x0izca+oSyIjQAotpDIIOpD4kX75sESAxHnvE3McwnkKqLsQdYjTAB8u38AAwbcFNjl6KrRAiWyXC1jILBEeA/JMXwjD6BVsEwRvPxEw1B9E3nJGG3eeT4H/6ZIppkOJKeYHDpcFu5hVV3fV1aWcqggOW/KCrLkMjc8yTbdOrwhPAbDaafNz/OXhQ3uiAOJIN9yGQr3CQjM0vjID3saPhy4yxJketE8NoM2vaMwiW1o82ESh8nL2aVtpaUiN2MFJNN1elOQoFFmZyPorUqs4Km5uAY7LHPHq4z514jbCGnUViO6oZFCizDGbdcP0FfvXOikkzduEYajVph6qvGM1Vo9oy8bHx8PjZ5NHyvv08WTiyZzNgYb3/0QBBtoXrCK4QTMBCDpD/ZJXw8cecIFIwcc8HL4KSX2qUQGx8HQsaQa0Z+HDh8CiLIG+4VxIsWjo11DOmUMKotmyBLiwU6Pv2NCD2AIqS8KMbIOhRXBtktwDTG6DsTAaBnUIIgXZ4TK7YenBMlYKycJ607D4imJCuWlaz1WKWwMsaXYNT5k+l2oydAa+Ltc+tB6SOH7gufRsLBOaxPR35NQSk40sJFar4NxUcyCBfSf1LnLYhgeXRHWX2HyB1mlnoUhrV2+dOdD+Mbg454ZH2ndcHqlfAe5Hbm3RxptGW7iLX8MJiL9cDq94o8Ir3Cgei1bk4g5fEiTniKKZm9y9P7e3bnc3vsza+xMbKdwNMssxCfgUOdDbs2bugWKZTqngES0gVFePXOwC0+pbIw2xMkPbAcC9HbI6wVeoluByRoE3Hh5/P5j0NeKuRqups/TaY9D25Lvbrlutt127rq1WmS1wGnLOh2AO3Oq54L4KECaTKWGc6c1YXQMhNTRj0uFlo3SNuNzXDPVB+3RZXm/yKD4etQ7eNk+GZToNtNlCKIBYUAbGiuUJ4JRXb395/vrVCyInqywtSA/UHuj48LvBcBIcoWa+98uoylasoozdAUjHhztmhei5SWNT9yF2r60C7X6CH+MnVstwHzIgKOg4MOICYJcmWZd1s8g/+4go4/ZaGPFvzxqidhVI2YVAe4e3E/SMcKFCi4q7/ViRc8fCsGjat5TEmbTGtHe0MiCBSdGc8RPyJIDqYNwJLnunEL7SjBC8Ob9IzC4lF6iklHKwf0dPng6Qx8MtgcdtHEJiBI/b0h9eEz6CLToVhC065ZZrEzAZnD3fblHYozWADSrLVMBIJTQn2C2Y4TmXtuyOWYbcKdr5TE0bFt3IA5Zjvb7Il5nDChkhN7pOcy63LXEzd9DDdtzJd3h/8v6WZWsyg6YwzKuM2YaRl3qmG707Rm9Dlo8S6lSiiZ4BxoqoPWF54z/E6R4HLnfTT2HbTICQaWXGqO5iDO5D/Q9c7n27pLBi4S7uXl7c7qH5ox0E36HM6kOTL5fkJoUOBv/0dhDq7uiFWD/qHzXCnS4AzchQ8RWp55uLXLRmmkcnUDEsLkpRap1uakRSMKHaMbZsQSqaUdtSLfmeIDDAaYwfB1/HI3eYSNRKEBcJP0Z9wjJQSRytjF4kGxk7/wm6fB1POnBHs5NW6LLiPdPkixtfn5AeqQDdGvT5cdvTL/1OFW43L2q0+jJ58GH1HFWk3nraCNYzAqHJKrTtMqkJyanAfkaVtRAh1sieoc6U2gZeu94sm9pLF+jGlxfS0MgzbWpDI6tlMzzOcvwJF7OTjquyhOEGRw4GrNWRGT2789TxeG3tGKLVPF+WU189FImRFo7cYh7EHV8RNsvqZcBVpRW10iR1IYKcelu6GmSZ9U3WRN4J2hdgKqSqxjPHsOLBbMvriBwWHG0D+yGhjpOaD0HQidkbzVsP1wSg6FsFFFI0idpjxPV68fEQ6pEHojLEo9fE+xxdDjzgs1ZpdYPCQUPn+5INQZcsygEiCI0swKxDqNiyqZ9gC+q8bhB/hsYXQJDR3fJfU20K0vGorqZw6z+FKGENCJKN7miGa7O2oTrPClH6Oy+ZuqL5SSsDDQjK3iJYm+3u9fjBaRFRA4j5DU09VAVVhnygtqCa1a68uCAA6K+JZnlddxXrpXM/p5Ux8QTidDJx+1mlzewiq+0EEaPpj4bb3j0sbWlyZSwuvFeFE4zkTLw3B8JDI+xQie+XZ+DCOt3xaTbOI0ea9jBB0dNzQFnvHmId3Djp89fi9H8+P/novT979+b9RyCUjxrNKgEzi0B5Vv50evYR/XPR2A2nlGCS7NGaUeMP7FiEDq3ILxEHTcyV7erlp9evvbPnf/fOTj+8f/f2w6nTn2X3nOWzY5mm03yZo8ENSL5eTW4emVZ3gVq+auaTCzPt4i5wjMxY0VX39avTF9775x8+PP/59AOO1RmabkPI+ma9XuawBmZHgv17LkSt0wof+wBw69np8xf/Sw3tLv3y6sXp25NT7+XzV68/nZ16L04/wq9do7qrc+0Ux+DJ1E1QruaE2uRtmQIOJZJex993XQ4ZGKW+Q1baroWu6vHAJQp2wyxxcPE/cxDQI5oWAb0qO64ONRDuYh/SHm2f0bu1rLBiaASN0d8YKH+LNGoG56E2NN2LBqGIDC0aV54eP/Y5erpo5gwLjw+PJ2ypEp9TOHfLeAlr51NZfxECIVyi34vdrfYJPDs9OX370Tv9Bf4l2FHiioxNWfB/idZ06HJItg3q2AEaWq8DY1fOaoS55nQ1zeawLJqXAe7UewPi5us3HruDADczq0qooV2ljKfn7GJTEHuNPjNXsAzopwKyxOwi2mEJ1h4YLYOtmFHYLOX4XugCzzdN+RFJZ/57VoX49AZpsTEaYf2Y/hV2WZeNWzUj7AAlVvLpylDS5sGM2SNlYp2V3C/inzIJgWjD7ogTFTqnW03En7wPSJs9dlqvcKlvKJjiBB0f0EKOq8T2cvJUQJdg4jiQutJ+oHW+ivSciNmIzVT/fw09QprroyME0KH1hu0aZFVChU2xWSUsetT+Y62zZ7d8Y3aoQ2N7NpKMbBv5xiDAF2uyNSFnHI8nEoRAUis5L6LPiD8IUfNELQbhk8dMA6ZI2GN6OabyQ/r30ZPHE8HD6DpoVFZm33XwwOcwnYNkCaxKjr6FrNUgA0Tw4+PvnlovReqo12uTDgduHnkmoh4jm9cIqr9nhQdtHFKr3jJf5Y1iEyf1LQomMySfugDGgFoTXzfmAnHXRNCgg5qIxxYw6tKkFfN4c4xckawoRoK1FBc5HOMitvDjP3wogwkiwixcgghFNlql9WUs38cqbZCXRXyI79Uk2hQ1oCb0Wz8UxcJV7HMDD7EIIMbNyj8OjvBBfoM0l67W/iovYqnDu261rWYKRREtYPbsqRJZh9KrcA2ofJ6voIWIIAjAE8aOLnSK3CGePFaGDBM4xPHA0wAFoPFjbCEJ3/v04ejoyeNHx8F/Ox5A8WFX0FSZ3ueE9kmF/AK2cYXxHtK0A6MBKwqJOtVOjzZEAcAeDSfI5heoDuWlEIlUQjISLYoRIxyaUfS4sjtuZIZXGSpyn0LfG+/v+WUOYkKeehIqBoh6kVd1A//C53JhA8pAoDhsMjp5l8AXfjp7DWURfUnNI0HlqDyp1+ksq8nYWVbAcqDAYldLDc1PEImQGiRsqteizvDxYBAqbi+ZAZlAH1p0gHsyIBmKFmF2kVbpDAThZLqZQ2U1/O+o14SOWIziGCI1bLvb7/SMD1VdbiqUxqTel6Jbta5m6qhgma9gmW+B8nhCPliJOLBF5Nq3yAAgkmu9vIleXv6arS8yoFUwiCo9x1eT29vWDkMX6GZPSgTUr2rIquxub6NQIoW7Otb41ez4c/7wEstnldRuDUxXkSYS2qo6JFPQL7Q5L2AxpyngIPRMTGTw2/EUqHkyrOOIoDe+C5ZHEv8CVRzJh0M9kKOiPSZWJG9uYhvv8QVloSGsHQtFWiLnN2K0QlMdArFZHPqszVbtL+F4ArChR4m+pK3IkuA2kOio9kA7IpoZT0dI25JgSJK6Qw9BOAzLjVV7GdTkQdx+00OcTtCbcG5kEV7JIz1Bb5XXJBkTDdnGeFqWIp9EIA0Niq11TV6zRFJxXLukpPDLLaDAjt4jlqlotaKDC+nLgfbu7HWUY5gkJzM6vvSTtBj0az6l0zEySElc0MhKEN8PVjprPGw/h9ZL6/Ts7N2ZujVe+yZ6DHFI9FEjwxd5RYcH1zVb5J9jxiIclHDIEWY4XjYsmIHTAcJHOBp8ZuMdZ9nnzZnnVWyqHykD0UHbrQsNCnLAUSj6jB4kHOwCGC4DYkpOzP7eAzIefjcJj4Wj2IYaGi+SSamkQvkRt3tm4qhbJiL6XSgRR7s7B5DaVGRWo2GPFTyJ6dgzU7vDZcNZEFMnlmWO6K/fbX+sV5TsomRHYAjUfMuXy6GUGF9OXEUhgA2uAbUDqAXtGaQ71Kx354mLmUecAcxmtaaXTZVnV+JPFSodB6mJ2ORWZsdCVw2opsGhtTG6fjRExjKgga6/tae6CV4kGDDNSm+onuQkNBvQMlf2Gac6W4XWTB5fUpfJIq0Iv/AbMywKxeB6/KY34uPzTG8tYwazQ7QFw87Oai3okCMwPs9sDMYjpSNL8TVA/WBgl3j/EiUcuKf6YNWxdPIAUQQfa2Ustb9A0Xl5M9Qw6VhLxFjNvRjXhT5IP7Tuw1ojw+vZGy3VTiFgdFfof0+be4jBxMgrY4zboaEnCxgnOodHjGj2MM3Go1KQuLu/o/UF8BW7jiuzhnVsAFZEt/HnsdM+EkyJSCR5jduePBq7+H1i+Sj9IxjJkjpD3busNNbeyQoPy+oNO+GthvWx2NkvnIY23gbK1sOI+Q7Cn8s7vb6hrNp4iAqe0Tq9QQ4jnOUSPRi7TJzfooHzacRT8HUbh8fDSfhkIAdRA5Nuqh+eNEPgaY4BoLohXQZQQXgtXXvCz3PTbM8X5xBtDWV9aShTiPVUtjiO2IyoB8YsahLOQf6GfJKEieA/rTOopzyftqyJsEhyKjXbsSMADdBRa1bURXw3tjkeALrpG92umWBgsh1zq09gPShk0TH/7MMYuoc6AWksESBIAPuSV4wBOm0YQqyA62I+GB43dNzQY3IoCGwcEUP/vq7UpDemyPBWXWEUfqbFZi1R0xTS34kdZ0+wgz2srrgHddlWGa1mWy3wtfhhxx1lFz/ctxUtdNBTgBDCXgcZK/rTN1YgQDM12kEq2Ao8aqEcubBrLhSrOfnCdD71nO137M0HO0uiRd6A6G8VXDrOL5X0GNQibsWfvOcU/UhZKby08Mo1a3ZCm7XiTVpdYjgf7BsS7SrUJvDPOK2GslpgSyS91puKFg7VFbCRNcEhLl6OVnXd5hGpuQ7F/qnxDuaZ8abl/IY0hzqtBxwg3Kl5XGURkkMSjfxK/ed//qf/lyGKacFfxt6vzeThr9Vffi386OFfAvoFBVSIzYVQ88W7j89fv/4n/Hr189t3Z6cnzz+cMthz60Pql39HGFS19lETtnblQCxijwqwgkVNYaP+OkQzIimt0PljHWjHVB9DB2Gn1W2of2m79C3ruJxo9TVp49fG8qkb4bD4QB9ot9+xNDoJ0Shhitj3PZDyiv3dSJm0xMB9j8p6NQDOKlWSSQHdbGlQtinrWOAMAcu1Z49vggcxz1ch/a3wu2ttz8nQ/dumhvOesyJV3e4ZKrbkjE/rj3uGAmw+HKTKLha/07DkigAyvPYoJlSTapmBdzbirnES/YRGdHUaMOATbrS1BpP+JZ3Xdjv75mlmmOuRCQ4Z9yA4Zr5yyhw0Z8Uq690b7AjzMeAj/twzMcNZSnYHHHVnkR7EW3mVCJ5phPM+duQDjJXMe0cgg28o9Q2uX0GK+tRwMtif9FLH49zdFzMzC6UTtHZgIqx5bDI+4Widh8I0J/7aJkkUAdZy6X+OY+elYV51NYnd3Zqvo+nlIzyUobR9UIZrphL8hJotGvrQOc3Wt8T1V9HNxbGDJHBuLg5pu49KI1RAWBM1VNJOQsYpyY9AVJMRlDPYZJXXNemLdScCcWu3ee1KkHBqJMyOJTQjIUWbRbOycvT2Vhs0T9JNnS4l5QOaM/vNj6HOFfOv2h9flhV3+fpN+BNQyufF/CcMCz8pi0WO0aduxPWo31SJAyKTHT+y80bc8o2h5Uz4S9SAHNRyi7FG0PvbPPuNhm2DYV8ahn0W0I4Bbo0DpDa6lrmy5g9QhxIExNtL51O/eZF8O80bNpVNiyk9JVQnQQ47VsXiW2W/bABw5uVmCswzN9yuiDwDYIdkTnXZCEX2pOOnZvwdq6uzwf8l6xi6GRNQfIO1iOldKOHrq3QNFFENB7ccD987GzHqMvyLsx6DP9ohNJ+4Sj9TDINYRb87fkyv5HP87eDZ0z7T7sIhEsIvW6+Xvqw3gvEdzldY+JFEkb89feW4FCp2e6Bj4BwJ9AX+tXj19v2nj8QoDjv+M2ZQIkCiJmbbHYTCLISlH39RVblENIZ5VpRYoxA9co+3E9JeF4Dktoy5qHVa3qDdCkARBJYlymG65VAXk01FE7TjoSdz0xGxO6zFuuM+AzY3ey/jMRv2MS0E9kWyIEWiGLOv1oBjbqv6Il1nY/IY2ZaheNgJYp4bzBjihAlQs6EJUG3HELTBTENfQuJE7ABcoIEL2nrUrvSjW459w0jMRg9FGFPXKN/WPSCUeTWmRkGpVC+Z6lXxmDn+Rio91VXmSTRdmzabIB5dXwMQ6nR5ExP9wexq2z90yCvY8RLFSG8moRgXgDGHvM9ap0l/bvv2SkNbcZ5Y77ztLQtGruPoXq36vRwQePiO/0GkBTPriNBFOx3wmJcgo6NsJzBtyEiSb51B9xuO0UiBMQ9kPAhpCsPJqEqvu7XnGakZTaWwvszXva4ieEjLumdaFklbqgZjGWFp+D8xt5bdhjfCjhPvMoZncmIF+SmeluWS3FjMcAJievGomlfkzEWWnLLuPaKGfcLYopw0He2z2O5hS1MQ44vAuur0gXiPq/GwT9m0DVl/6HgYsAWScn2/09KZ3Lbb9BDfUMh3dTMUvxStQnOUKdihq0shvago/3xL2sKPQHj519+yG/oRaJUgN260/rHW++3dNMM/E5urmxD2/IGNFRTW19STuLNW0FuntoPHZN4d9ZJJYkZfu5ErEpzJzBHZYDscJ70zCXOMf2E5zaom3cuPk94OcQDs3nlzgVzIv4kz/4COlXDuTsS3jvH+PZlyOyD7Uxj0dvbUP8p903wxvcMy4eRnqC8zzmB7eett3rR/iv+OPh22wTCTJpSHcaAgPnQRZJf9B/GTffke0Y8wO+Rci9oz3SaeVndxqy1ONHz4cFNgENV/IUv6J++9cDhOolNtBSmrOekzS1GAGj3HLMe5T2+gTV42RD/IXptUwmhftcatdUsLY3hZ/MnMpZTLRFth1TM4RuBuNP11OEnX+kNr2eUcgxYbuIcn3deSwokLTVDuke6cqHuwqds4kjFJYriODjdzN0cbAsXQITcuNaQVC0LzzXn5Y2fgdAQM2/MvsksYGLHLUbM8B4E4qstFg9FRh8fBeDBh2c4PohnmLYFWyXncwdyGVrNycYxV2bH2ScuvPaeux/kkIHcYM519tHxr1mEnsIPa7FLpvRvCWh+1vStMx2VbnDCp4e6NCbuBLMN+Vq9vQLi07SXFpDD/IvkzyfYItRn3ona28i8cjWcdE91UhBKu4GQux2ztmKUskEi+vOZazCeuo3qzWOSfOYkFe6WFipRFYTsJJDxOc8pj4eRQnufpeVHWTT7bnUdZcib3JRw2te+fUFlau19eZTJ/qzvTK78w47Apll0XuD+WXZmJ2r8pv/Kzwd3JlZ0y98n2SzX2EVxob07+vpRL3S4Np6wIdC5en2eBlpY9SZ1xaDrfsky7PxPzk8G98jCbUT4ZuIMzkXk1Xk/hiWbbpGSmxNJkE2oFaHY9yW7vTsbswIoTe2sTMm9snmgOJO5k+uVDDtgpMRFNPmcA1l6+BPGXeTE3IS+Ov+8qLzZNVsfffrdLDWf54b4A0Z5siL1ZE3XjvOnMO4kuBUptMFGBlh/i40FYz0pUq+rM2JgzU0mI8/MG9VwNpmyqbni1cH2F8WF6R9kiMeyIcBLKEitKSanjwEjnU3Nk0Fb2Ue47Vk4O8GcDZSJVr4G/3IWt9m6vbCxnrRdBwripsK6Bk9HrMTzoLIA+lW6q9J4uMfLqEHOYsv+7twLO2uPEHb25znUs1YaS41/pPNVuCnTnYOis1CDG8nHYsYs/HO+2vh1Kph4UaokdByhaznmgU+CGyzrHpnQHDKI/xH1I5oPsqBTa2UYL/FrZSRBCugbJdnFg8dh02X79w3FvXndaRd093lgjicEJ5ugiAuyWTjFdAhPVGRDYbx8HI+bQgCJtUoypbr3l1Hr2E+bmw893+cVLjCX5kbcC9xErwNJoX3eNGigLFeAKgD3hdUXM72MjbIAXpUJBzl+7fANmxGbUkBCPchydrYtzhtzUt4/Dr4ytOPC0mnRHzW1zVagknmJXlbqaHe0Ivwgcc7/13L8r4gBH2UasatjRNCpHpwzrgdYOI2dYBMzVNJ9JjCRbMQHzZvMbV7mMAjf5UNLtLOyqwsEyNlE1csVO1moziO4x5m57PlB7Jnd9QghLDX31bCDwrmXTw1a6eo1HWL4lRzInBb7ajX9F2dR3Q0MoaA2wmojNSDabckOOQw1yehLeiFjmqR6gvT9KIasNYznPKopejXdckzKSpIsUGUAxFWMTsa8o6QQ6CZngpnU7GL/WPktEj6DA//3f/0e1UxESt1Bea35hqJNTuEkmeqM+esM+7sw9welCJf3SHIPGTeO0Yq3EbfJRmEDnu14eG80AZZ3V7MPalKvDWZ95mTFKJh5Z20/MMeO8LhQFIl7WjU3ryHuiP8AgrUeTfJOgcGKEVLoky5HxBjPpQA50xHhs81xzpJAJS6cW6B9Dh508XiZUJbBJxDYrv+nLs8YRtyRhcY3e5BvbrNTWm3BP1mXkJsSZORa6LjLivugXQE3xYOQeefELHYx0ZjHKrnMZD8QVVHsv6oRCI0nKowPbxQCmffmUCWaRhdWJfhBQ0VRkXZWJJVYhS7zmtbZ2EwzoCXaDyH2LhHf3ZOOddScmMb+klIB+wvY7onUB5k3T2e5w4wXU6FiZte4onrcIpsNgB5ryxl3K23YmMW2EEovZnRpNo9bmUUpvf484CRsfEUhUtOAhfYQ0tthd98dYmLCHTwfDKQKI1LmD+YJPP8btMkMNQOaqDrokUIOTw1AZmBqZHv9IGFav5bSDzYYOLuvLeH27P6ygsz3t5K0255MdKb1qIduQ2UlYEItiHsFqPTpuu+9rj4mvCvhsjdNxk295+OmQQCc2UlvNR93SJqXXlk582wdfn5hIq933+Id02KcdEwh6MdiBjeZx+Op3m+bd4g1lZeaMnzTAzsZh5mZYw6Rc6ATOjjFL0R2VDJeMF2p9cogzc0jdNebdSKdVTupECWjx+hzQ8dpM6BEDprnHyHuJyRroOk16MQK2PC3wUhxWHhzpC888PDg5XuxEl2aifgB6ci976bUkuv0bY6K1LwatwCe932JavGeoQc+2cM7SHSdFRzjx34RzIOuS7QzEXdf1drBTby7W/ZE3UgXKIK0ApIXpnECInS03c1xazq/o5JnBa58uMAUbqUIdo841XjgCG7G8AVqYLxvYOEJbNsKJO9HIEUDnsCkPM2wmX2eYtNpD155idhMpZ2jdbJRs4b2DXeREDjZk4Y7iEs2AfAFmoDVrTxnsdKSDzUcqvEH78dExEY9dWWeZ0A2EXmxDyI/Gote59oUt65Lj7olbEy0kVco3lwKaOuqa2PDWs8h7JdnDUDhYlnBWOYsc96qVJhLfpzTH3J6a8QnqSkj3GDGfjnZ7mMLRnYeskfi0OGNmwbY+0vZ3I18g3gRKTLin7xZOgNyKbI7sJhVpAaJNXOLY8LjDNpdnOGfD56H8xXMxdXAaW+n5lAleAuQyLdNq/grPXLVZkzdVu5tcf3I5SvSQ85wv0B4tVFnAXDcVXpwCsh/vYi86Vtsck9tRy0GBUkqOaBt3R11tjVswlh7yXXhxezyozP06tNcZcitOi3mEbldkKzKcdm1gXBPkVjCaodJOoKjhvzu2JxvZprvkSu5ahSZrp8jQWpsn0VEAhaFC8sVcYeR9oGpDWBa3FYU3gKLCc1MQOUR1qqaDTcnidha1RW2+L3gGx8VvKcrrUCgn59akwvbepVYeIY4OuqSLttB7Gii68QZJZwCM6ewmoS7QIWbGgdbwAwnQgt4nNQcMIKez2NSc8gba+XyXVo81Yfc3IWEw7VcakEbaBYvTN7qrQ1SOYi2ot30qaXZSmDGVNLwJYl3WRmudkdbusFoftSY6fAc3B+UR2iS+PnqPZlPUM18wdVq2GrpqkNbsvj5XKV9mwpaZXQYrXhd9Cw7IjoxdD3W9/deCOjyjE6SytQG23QexaRhPpqv0ePX25N2b969PP55iPjarp1BHrDfjC9BolzToIn2Zb2ZEH0ZCUpArIkNLPK660+QkSM4yaN3GZHQTjxECx+5nx+5ILU5GlIx9zIk7WiVtaEinig02ouehM0pz4kI+ceHiOEziO86hf0Oua6FYXNlqGP6eVWUyz8WvfkAIm05p3D22vfUfjQ2bgzktnoTDSdtrghzkcJsmeDeUzvthiKymzDTFIJarb1kbihputA7BEiS0e3gPIHsocHG6MpC2Uw3bOEnGSprfGUxwcQyCLrl2FOsIE7/7i+OA1VLK6IKxRkSMsm9vDAhdLhTOx2aFSu7tyZNfe2u9ULmMf7mXNWrdUR+jhl9oBYdflNkwPTjzAh1H0N8Ct1F/5Cf+YuezOMY3bIrNeW/aWUgkHPGWh8GYnLn9oXrNV/K2/NrkwvOXOuHV4VbCKxu07s1LAtAbtN1SOKpGcRjAB1zcy9NfTs+YEkWEVJxj62ZLvxy6kNJJ/mGS/RbK2XFna83tYJi9Iziwfp1Dcig7yxD8mS+UUCpWWY+Apub/2GQgt2C/GGNpGEe+aKPc1J6A4ZEJsK24vYjgG38SmvZlbWudYrZ9O5sROEaObMLVQ8lYppNL8njxgkX64VjE2W2UqLs1NiRcrEPpxZ78AZeeneXqjloapk+8PwhrG+SvBfeyHOPxfQPsa3e+ySiqrGVSntVXdxFy8gTZYU3GSe2+V9s6KPS7fUj67/huHkc7d8r9CWjSMI4yFDErTbmeMSsnY7trqHEmkbQwGFQoUOR7NhDUtaOSjahW0g0ZlqAX4+u4y55EIdxsUcdiLcaCp6Daqk1tYnFSbW0xEvtsLv0JaB0iJKrIvblkjQHx69J9WaWf04vbZpvdyZ18f+6oeVO+3HKDFeaSirvSOkLjuD0QS5qpIiDT87K6iX3XUmrTDIjYrYwqQJy/OwpDm7uYRtU2jJjLTTgehYVidbdyodOKzFFCQg22TjBUN5GLgnixqJzArBlpWSUsunBgCxZP5EzNlcTm4EubE8i5x4o5snsllDNZxId7F198GB0/4WHnshk9YTRY0w/O/YQKd9k0+CK/bKqibv4HvHNC+pIL4mm1tE+mTQbZukirb+C2uuM3LE3biG+Qa6t1w5o55moEePfnvtCrISFUOpIXhYkhjoT8GB1rvv3ing3dDq8T9EJEAFgCZdYOeQTk1dStJql08dSBg1F8XXYXyQMioYyjbvsiUdQX0hV5SpI1G71dFUO16AVs9t/p0V+E5BxG2J3TOXO348EkEHoqd+KhepLMfO5LliGpRmBYEKCy+MKrL7KssZnkDetQDx06zXN1EPn9SL0Q9rpLwcWtbVOwJT6hnLHL9Carap9Oflg42USOf4iLH0hdxa7tYj+TBE5VRBX3ZQOgAhzxp+VNEgq5OURgMB3sOxii/Mjia3JepXOmnltVnDzKe2ujfoErT/HaFNtA7xTGh8VwMhQiRr1Rrft04Thnw0pSNYBcvkVPRoq3FwFSIh8+RSEb6DOX2MbhjGxWIEAj47z0g33LxIJzayCB3GL7dQ0GtwwRQNthJkWWyAJZzsYBkfhYOxCu82XZKOPl+GQQOn5MdZOtoexgMAjMfdZEuufUzSH0g+oK46nH5q8ZeqxoFw/bmod2SDaP8l3WOZsJ5t6zp0fPnnrkXEVcJrN7Hq0wJacvF801cN4iUdDIWzqSA53CxnMYbMnRZ5sB2R85ZeGDtJhxCHyWKSO3Yrsej+G2SuvfEd0UfqSQn4r/QtfPq/MN8nq1/nIiGd1CzDJ2Ui6XKYzn74AE33NearRck6dbz6giDiiqkk2TL80YMdcvAYHdlP1KusUxi8IH3Uisxnrf7WXf+zLndOHLOA92P/Q6SJ5YeCKtNKAn49LoeBXit/Os0r7TBDHm5kiBeUrI3JuE5uSixDRGVA5HRgWDAwYTolwkZ/gsaNDbhAqLPtF10uLONE9vy2Ob9tpsmMZWu5aN/wONO5XaPdhsXzwFy9lK3hwZguXTeV1eAu1/WzYv0THC8Z+1x4rOLcrFsr6ICNh6DzgbKi3h3H7IMg851jenpJEzx/XwUJDAYeeURt4LVgvUmykgkgaQk+d6tkVmOxwdr5lbMILJOB9kagCc0sT9FcPAYhGSiu/y9NdN3x6YGwY03mJAEtdboIalltCyIxSoibPhhcsr9BeHk4fm6IOtE0G5WbfOD8YPHcO/sHezLOfgZFqK4Oj4KV3f0lQkAN01Bap0O7q6V+ErmiqJFVj+z1c6CZLPL/5JL4I/6wXsOWYa9TlAe2RdHSWrEuDqZbp2DzLAPR1Io13nyf4QP3vadzUSHWIC0hkx1YgpOEqfzrUBZGWsLNRLtMHEZ34glB49fFgxzL2Fyg5aPsALeP1XOD1lRUTQ0qAa7w5DVQQrQ9Bi00rzefb8Z3ughMdme6q+xp3B1O79kXaANax9y5PZTAr5aykJjCZO7JEiIiDWSMsTBNoXOlpdYvgSMBd0Ew2xO4QUkvJS+2HeN4hWht8f0xoceF8dKXtXw4Y9i/ewyO7sRXNm6rVdtcV7l/4IRCSiR96GB/2FAYILb7lTW2TVqrpVTuMuFLb+NedsWTKH+oNI2+O0LuUC1wE9W5ezC7wigIZBzAJJ9inqgB9nh9+xf7T2xMZYGEW3MyQ1wAW0ESpkcXNUZqH6dbVZsj8zfuGra5J5NktByo4Gx9SJ4zuNKSlbaA/7PYeyijAgxhd1kGJrZowmlQACOYWbbQ73+Hyjz3O/nzMhI+0WYiwRWs/YvWrpuTjUCuyidVZqsknCXLDUfn3gSa4sxL3IBKIhi52yiK2LyONSPpDScTton70TKAiSY+owIEoXhN2ScibU0dzPQhVNSeAcUbepW83Ze7qKTdFxPhlhidgJQbZ6mspxeuMI6nuFI2OGAdIiUto5ubVuIlajiJKXYy+iwjGriMV7XZX18mpv5fDeqFFuGh50fIk58QkIQQ1MYo0OSr/ThUIZ0SvjjLAVAYYWBeHydWA623oNDZim1Ug3wFSp9vCWZ4rRQq+GoyXe8sTXT0UaUtB35nnzE/kS+x0xoh9AAnPl654QaayFKdAJkU2z85xhJEyrc7r3s8nIE7Eql+HDh5fX+Dro86XQs3OcO+0Vz3x967KckttCttaXO9N7BAo+xTwIgmUH+ToI1Hwi7Nsnpcfm5bhfip84k8Ze/9Cc84XX74XyILZUebhjZb5iRbpj1Xf33D1SUqjpgdCdtK2Ba8/TP75HRCzkLf3WtoCtBviKH7kRqP0p7ja7a83o69eA0s5h7rvDmfy4HTfuPr9MMzL26k+0B/a+MQbGPxk2ClHGZjmnHMiEMEadD87cy7UosEx+ASlr4WJZnu8ECbpyTxJ13OfoioZg/3pLoZgCTGFzv9w6UOrYwrHwv+FY6dPfc7CcT3etvgxP2I5s57j05O5eNAcrGZvdvdZNq2G6S4fw8C+iXpctYh+yfQOypV23WYfbqskGZx4P5crurfUlMRRHF2/pu3xJDoUXp+zqJAjpfjvC/cyKxnjBl8OFxsSEupxkLIwkek1ISkahYIYvjb8Nna8k/jsfvw97WVYRxb8NF+vjpywdoafTFU2VGdNY86XyFstv8aNyI3qnhrztrXHAIV4Ja4k5Ld9jPMHnGg3WFBoOXHsMTDuyh4l+YAM/VI3HZA2jNJfTDC/MJDth2iDl4OmIgweI+k6J2PGVOKcQjyrJayjQoLVFL4NZMTt4c4Ve/7eEoRZkrA2lQ4MCVYqsODFXtzQLHCx0hzCAue2Rhg8kC5GkCvXXNluH668jCkx/HbEhExMtryPHiB1B55K7RXsljQfhcfh4EsJSV+hUyu2pLTenW5Eyafr015cIKRIuaVZ0Thns5syox212nvkXgpPeAsTFAPM4Y/cJ9v+xXDauOe3zTBTE8Q5tsW+rcDK+BOC5yeHkJ+Ui/p6j9DDnqsZB8hzOhH+EdWmxlwEHM7ODS9yjWvZ3HuagLT25nzr2do7EIHHIhOX1xtFZW8EdAXWyY6wg99nzlZMiO7hOT0s8CnWdLbT2Y9z+ZHkVRME87LZMN+rgZTjPuKvsrSuxfKQIpsXDtwHFDXEnePg5CRF/GtnEOfTN0Y9wgf44ho7AGnbVGyJB75w2iPaIFlzy5qgU2tUcFGMLhbgaeX3RjhLjGNa9LvScyWxTuWYdGq1141faBUoTGvNJe0W3I4olRrbtXt2GHh30YCeg/amJhf4DTtV7nfv/gFs/efNzD2zAwdCXunGCGaw20W7NPRz1v9JFvwvWht3QXvr38cBvu8G3HeBHmPhFIIvjUF1Peq0+wbYO/h+I4kdr'))
expected_runtime_sha256 = '902e8bafd72767546bc006b9fbbc6169f447bc40b4fb4bff3e9aad9f32afef06'
assert hashlib.sha256(patch_source).hexdigest() == expected_runtime_sha256
manifest_path = PROJECT / 'manifest.json'
manifest = json.loads(manifest_path.read_text())
if manifest['files']['src/fever_colab_runtime.py'] not in ['f2ca840db6e6d871bb42a1bd3ab84ef77613d7946541add86c407b7c4b26ad55', '902e8bafd72767546bc006b9fbbc6169f447bc40b4fb4bff3e9aad9f32afef06']:
    raise RuntimeError('Unexpected bundle revision; use the original supplied ZIP.')
(PROJECT / 'src/fever_colab_runtime.py').write_bytes(patch_source)
manifest['files']['src/fever_colab_runtime.py'] = expected_runtime_sha256
manifest['notebook_patch'] = {'revision': 'colab-debug-v5', 'runtime_sha256': expected_runtime_sha256}
manifest_path.write_text(json.dumps(manifest, indent=2))
importlib.invalidate_caches()
fever = importlib.reload(fever)
assert fever.RUNTIME_REVISION == 'colab-debug-v5'
EVAL_ROWS, EVAL_LABELS = fever.verify_bundle(PROJECT)
print('Debug runtime ready. The original ZIP and earlier experiment results were preserved.')

Mounted at /content/drive
Found your data: /content/drive/MyDrive/FEVER_Project/FEVER_Colab_Bundle.zip
Verified bundle: 19,998 evaluation claims; labels separate; pilot training/validation disjoint.
Project ready: /content/fever_colab_project
Verified bundle: 19,998 evaluation claims; labels separate; pilot training/validation disjoint.
Debug runtime ready. The original ZIP and earlier experiment results were preserved.


In [4]:
# Install the checksummed paired-evaluation adapter.
import base64, hashlib, importlib, zlib
paired_source = zlib.decompress(base64.b64decode('eJylWnuP28YR/1+fYuOiWNLmUae7JGio0oDjnIugcWo4TopCEAiKXJ22R5EMl7yzcrjv3pnZB0mJst02QKzjPmfn+ZvZffbs2btUNiJnVSNvZZkWcyUKkbXQ8r76/ub9h5SJ+7To0lZWJXuQ7Y5VpWBql+KkQt6Li/ev/sayqmzFxzaczX6uWNukspTlbcjgY1/looDVWZ62Kcurh7Ko0lzBKkzu66ppQ/ZhJ1i13cpMpgUr0g2M38pCMKlmjUiBtLI4sM2BNQLHw8JLlu1EdldXsmyZphep65RQbvOLuqoKBqTLnGgPZ8+ePZvpLdm/VVUG+7TdBa3ci9m2qfashs9CbgxV7B182uFbcS+aJKuAtqTpSpzDUgWHbsRsloutoSHpd/MsGUlTVa0fzdioIcbVj8Yss7TMcbpQ8Wq9rJuqrWBL/JixLTBwU1TZnWKyZKtFcBVcr2FV6Chy0cSjpeYeb6qNaNo02XZFkRSpavkL1TaeXsL3YaJdP0ZehCQTz9OrzbntDLGT+yGKIUEBezQ3q/Z1IVoxOdd2npmrWjhhgsyO3QwnTMWB9v7zQlNtV1xx1Ypa8bXvzzmdGKRC6+m97OpDsvrtjgmRW8cDWDirasHXX8UcOcZRX/tO2iopsXux+Pb6m2/H3b3UzZivv7n+GmXDYJ4S7DcwH3HTNFXj8V9L8bHW5oUbXVi5gfaBZuMa/IS2rtw2QvwBo/h6xVGYiZYjbmaUAgiik674bVFt0iJBTmH/Ce/OEPbBEgKWnBMhczMVLWsvFVhLtiPien8AutmQajaolYaAorpNdlK1VXPgazwJxwnJPs2aKtkuOA5tQKXZRqg23qcfvcGCwZ04xEW63+Qpa6JmdTR3bZhTVi1D8w2lyopKCQ/XOhkcGIqwM9mLtpEZNKYblbSg9wtxsbjyz7Dje5gycB5Mzwb3BR7GbJ7tmD2moYqMerQn+r6k12egPyzTvQA1O1FyfQIttk9SNXB9R+QM6J2irFcFIm6gtlMUfvFhztD6eqA9ok3J+2s3qRjEAbndikaUw+MQsVuMQc4z0Bdf0v7G0006jS8mFp2Xc7NhWteizL3HkVFF+jcY2rXTqWha0ZAFw/Ggk2o0lhrWAbfBVdtnNBR6oE9LvopHqBL07evF6VT9eQajBszxYQPydxCer775lkcYoUL4SDCeDnyhWdRFDetszkw8ExaebCxIm3RTiDiXWevZMRDPXE9YV7U39GN+H96sEPrRsKox8bQovDqO3djV5ZrcTY0+xLX60SfcmRItIgZlNA6czgEcHENaqj9ECXDjIBqFqvcgS4gn5JB6FRn7I69ZTWrFOrhojjwzhphGtF1TskfuIIr1QhE/B1MYLXrxZrFkrQTr3qb3cOCteADaHdFmExKi/o/3JIPwevK5PhWP9G/ALc5KeqebAG7KQYMgllo6efQmLZR4ms0yOJVif3v3681+I3LQAo9U471QMgfzta3osQBf/QImxt7CwX56y9rqTpTyD9CbvUhBXHBOgm5v3l1fsde//vAKnEa7g2O1O+h+/e5XJj6KrNNYbfaDuJcZoCyQlaq6Bv6k/WR7YIA8gbVARg5RNEVvUoqHIUp1asp+2YMKzbYABVoKstprGt+TIXsJ+TWwWdUpTQSdiFQGN6rBcCWqMMFHhmAvSeCAbZJ4wK5tYJ0b+WzV1cAhP3RDXO8SB4fkluL+z7CtAO90eYoKiGsDVdCuV0aoogI4zG0jlIp/BtxNmxhQWnb7+oDnL2tohCFd0RJWJBMBW4cxGGrT8lZ4l0EhAJTiin5wbcLeBqNGTI0rGh/Rvy+ur3AVNMK0PHg4kSh2Ag0NlR+DNAdPBngGFYq6Vfyh6YTvvwQvQnR8RBpoI73nRKS4sRxnSApoQSZETqJBXYeFLmhpSDb2UgcJw6Y8HtPl0T5BndJqREnQAl7PSC1iUulAGyXgwFJVjYp5DUuOxMB0lmOWbjIAGaXRFwornjnIToLilwNhes+fG7r8kHyBHqER6nKfqrvY9K942raozeRE1B1fh12pfu/AOwrvYkFEsHswxaqJPb3Kcxznh6rbewt/jh/m7xAsdF97e1nGZqJWBetXB+coy3AL3MBt0yIsqwasA/jm6Z2COr4KcrmPcc26Ay0mDYNfOMuhFh4nO7q+4gSfUT1IXeL4klmFA9V5Gfeahu0edczn11cvFv6fF5cwXPOP/Ah4ebk9eFbJA+7MLyFh8kA3gKzhhJ7dZKDNftBWLaoffg329skoyAGXdQioFrRAlPC/p/nj68wNEjsgIHFx+UHI212rPOcKUdomNLi2lXWr66XGK4Q/dNtqGMbXPmGXcf9kKEdkqydaPxcTgwbjbIfBBRoVfX40DsPwRgdLEAQBbMcfsk/6A0x0vDcFYOgBS88VmoPn8VClW2HMhgc83EiAAP7aBevhBih49B3jVWlXzFmGhIe3ovWwwx/RM1yNDO7Ub7zBxc1AF9/JoUNYGYTUC1fRGGNNox1jGrVWgNMAschGeU31oAJMqoOqawOnAYHT0wD0ssOcffGXy0DtqofEeWxyQDsBKrsRaQs6hrmViq8uAwAZCQStBGxX7rVrD0CAdde7eePlyXKBVlc00LUCIEZ/wh9wFIj4VSkz0ERSA8p0kXIaPecYgeaEQ2VZd60i/FZwDbTQZHCsD5nrd9999xeUHX5D7mhXxSYc9gjaK3O+7nM+HPhkZ06LydSXBvG5Eb930KZITuJjmrWUDLPFdwHtb0tBmlhDJqoTgDmvgd0ekYqAE/fgF0LcHZguZtBZAQDoaUzfFD4c5OEDwrYgpg6C/laKIrcbG/n+FfzW6Tq/gJxx5qbLQY/ZvoMEbUOoAXLoe2GXGImbSZ20oaQJ4Hj4JZUswb1BlPHGwwNQV/Kk4+a/LibPBd+p2x6CmCaPpugAiivhzkiaDgv7tOywYgBR1/v6yl/qVgyHw64EwTh0z5jzlc73fN6Jzlx9UZd+tF729ji3BStAYjmGWj5jxlW5WpZboUlvv1lcoeBNT1/gmnDRx8D8BbcLgO9CYmAPnDoi0A8mVxq6dpjtnED8CLHfeQTeA4acYCzgfUIXARXHQJuSHCMqNNuQOoDyiOZrspkEoGPdYo1rMitL6CdJ/CegxLDNZXImHuiyrGslf9D7Lh+VU/sdVEpSSAEgybSF56bNDCaPORaA2T/lnaxFLlN+bj3+5ua3m/f9QKbKtAZn2YKYHcWPpgoXmeM7lSEjmfLntkINGp6S/xi5D+C9KKp6T1UGshliMye3coann/aXAR9kSH0swEUJAULuRTnwH7KmkKaC1dmwPKluA6OCDFIvDFYPvvcW031QWiRB5Dz6Gv46ETqPTpoGo7TMgHD6pcW/UMFgkfJeNlWJvDSjBi2ephS8GArvZ0jGIF+8cOrC8IKg97JLcn5bvDlQMBlJG/QGOvEvIdltqw4Yl0NWoNqQP6GxyuYI8FBFQWAUnJPWgIhdAYIA1a1o6gaLRPEjFatOZ4GmSZKj4nMcEjxa83NVEbsnj0YkoDmbYimPcOrTGMOQCgDdtOdDI8HuK/TvtO+gjuA0SddVgqHbRAyPDma1Xtr6XZJVwJxYlrSE+brUdwQQ72zYi6jIL5DHfVo4Io1wOLnjaU7MPfSLsJaO+uA5TeHHwH+qqouPUoGrN0lR26TZqP4+VXmnyTSSwB8uDkHd7YMRatCro7wdoL+Ox5xKCsYPpa+hp6ZxAqP8kt6DovUMOK6sIlAxTsnZmj6J4bDNtogmU3RFxGS6/Ti+iizXdNPqcr3qbdPRDfDYDlhMDziXRr8zVYziwJQ+jibYxACLkJ30MJ43gx0wTe0gQMYxf3/z6od/8R5F2VOQIx7SDxAab2Fy7SePiT/u/W8pt/MNEBwdYGwML+LFcmQPL2LEi1SU24iCDvXjz7+9+unHHyaOhXcaEKUiY2tWlqBvfSnS9EWkTh7/vmp3/VWJAmaijh+gCbCQCUYEZnXgCbm/tMkoWt6M7QFZY3qvq2ld+VZ/a99gOufGDxmYA/DhuMFh98DyI4/HnEHPbDgz9hhHucroayJpOWkB1uhqRkMpy1IHQPRTlKQDIXhNGtai2eoNsTC21EoWw0Gw6MiBKSn4ipirMY6GZnT93AkVHJFI7+LLJQBg9HVtc0BtMowKIdKjVNDlYOmgLyIQVTzQdTfXjKpZV3wdaHgWG3RGoCzuyxwz5o44LIAOcZBzqgE6OlSqPvYTttR+8QydVFyVkG00PHDKFFMybPjZV8dRTbTYf0O0DXPItwaACXxLiPH+VleNoU/I4cLI6GVsspvn315GG2SxmfOZjAW6XsbjMZEVJex0MZV3OJm6fepdqsSkmhzxi5ZJDM3AKvoEhXYhQzeNIoTxEuT7xmAYOIVG02f1RsiDnN6ZxYiQz8Hl/38Dqm0eu3xNvZK3JWWnntnGX6I30KG9V0P93GKkh1qTTID+X8P9mXj/pQHfRfwzMe+kcSK3fU36M4zShhMmMChDKfnyIXVULoh6ZTEqEg3VJTCRInJxAobBMRCSd21mQG9ZPWiwa9K0U4B4CjqCiRNHJ03BSSCOzOlW3EbmJ8fKk66vbNCOLJeRZxD37MMcgkm6MeA3ZivkJV666eJHCmA/NzzUTNQIq6vxLsnTFxaG+6TfZu1zymwAEijpCTqaPEMPPKgoghPPgQzsmkIY9DDDztN5nr6WGM0ad5y9l3hLdgPKlZPza0QmMNvVEsOqmMMnpENzK0C6xtB8/BMuSFdxbIOIwVonwZmNAKsFGXSbQqodJkpC0oUYXmf1Wh6O7JtYeBxnZOMPaurHmcaXG3owEJDBDF8Co3D/iShHudJw8ynHbdPg2InNbOUy4l5utmdUK9EmYWWSWI9Ptde+eawo8XndGx2E0Kj3v7kPVHHDRFMDeffqx/fJ63+8fffTzYcbV/HhD4AWLagC850I1hQnn87AXlCWIfo1QGn8+WJxQs2lscLxwJfxtXG87/UzN3udvoOMGS1W0Q3tvXBRitwGRCW1hKXwKrA1GJ5UySk53rns6eZ/Zox/lMua+rFFh7a3x4e99ShbLMACUXkKtfmMbhDrlv1dHDZV2uQ/Ihubrm4j80AMt5C2cYhCAZOiRrsePJOlhdRcn8PeG+uDapse6vmSWOjouKEfdLGpwrYhGcbpWgrong2G4A0yWmqSvOAR06VNbD4GRnhPYZYIhN0m1ov5lgwqWhaHyIBEhxA0OyPt1/UzqiHkJVRvrhMKWFEMcJkerQ8RmO2wkke5kSn4GwwY8cCM40seHCUnfG6KdgZnureDg/sZnTDpaxl6AOo5XOVuh57hM9amxaqfDUaI0O4l3d0HkFa0mg3mIZeydxD64sFlypC8pbcipKv+oysWM9GAMfz3iI7ALElvLvU1uLtfMZP9r+KryBzLNLlS6vApZV/Omn/qJebqpLZnXoqa12rS7RKiyQiD2nTbxORY/zEYon8TquytY/4Tvc21LMRnuUt88lAXkpJ2pYB5LJNtapJi4C9GTuJWrq/V2+FbX4ULtUwXh51HoeFNCIZMPlWNWNPfgQ3LtHrg9ItT0jQFIBB9COhdA4G85dFlwF0JFk8y6nHV2JMeWuShqcCVRZdPo6Lb0dXbNKXjgjKKIw028epU+l8ctMfnPa3yrbXHTc8g7810kekT93hGTAZ0s7xr6KEVHN86eMOwJI3TAXTQQlqNj4FlTT14g6R8ZjCsjQ+yxqI0WZnekbCjW9JE3mlBj+fpkdOCH460aw4UgU6MSrYC4tYQjWf2Oc4jL3lEHi5wlxvkcaYuKYKTupfjNR5KP4LRBZLx0y9XhtJVaOXqRJoqzCTw9/xVgL69MI/TYLz5Y7hJmmUdRPRD0r+eSgCmZEA3GH1iHmJGi8vL59bbUfqjALkBdLPT+fpi3Hs57vXPlMu9/kj4TFCC5ic6LBJgwn+SEvzbAMvaZx4mLJGF5d2+VuYBSCBLVPf4yne1OeuT/wP2FVzl'))
assert hashlib.sha256(paired_source).hexdigest() == '37b680e7eab0c186ab01d3237e34cb59c963d08f817b8a22967327df4422c24f'
(PROJECT / 'src/paired_roberta_evaluation.py').write_bytes(paired_source)
import paired_roberta_evaluation as paired
paired = importlib.reload(paired)
print('Paired evaluation adapter verified.')

Paired evaluation adapter verified.


## 3b. Verify and connect FEVER Wikipedia source data
The ZIP is already in Drive; no paths need typing. This verifies source and retrieval-input hashes before use.


In [5]:
import base64, hashlib, importlib, zlib, zipfile, json
snapshot_source = zlib.decompress(base64.b64decode('eJydWVtz28YVfuev2LgPC1QQKCWTtAMOknFsOcmMa6eUJ5kOw2JWwJLcCgQQ7EIy49F/7zlnLwQpSnHrBwt7P9fvfLt88eLFm6Gu2ZurX67m7Fd1qzpZKcF0Izq9aQ07/5Z1sj8va6G2zMht1/ai37E3L3+6vp5e//OtMpLNX/6QTibvWvbjhw8/s1LUtWaiqVjTsnVbV+fyTlWyKSWr2/Z26FL2YSOZ2m4HI25qyXQ79DBYtn03aKY0q4QRs0nb1LvR2XeyNG2vp/dtf6uaNUzvlIRzeskqWUsjq5RdiztZMdOLEkZ6aYRqmB66rlaymnRCa7GWVjR3Zte3d7IRIFsC0hq2Qlv4E0RvVFnDgrb3p6eTFy9eTBRawbCN0Jta3ST/0W2T9DLRv9dgja8StNJK1TIxaisnq77dsk4YnMrcyp+haQeGvob+tBO9ln7096E10h+ykneyL8q2FjdFPzS4JRMajSUnk0quYOu1LGCbCD/ibMJWIK5pb2WT3Il6gF0btoj4+dv59+c84RGPE2jNbSu2rbfX1Fq4Mdta2tar92/fv8N2xuNlhofk+F/ay64GO0ejo+IJGn3oG8Y3xnQ6m05lk977oErbfj3F1pSfkY4kcqLFSuacx5MJOFprdu1C7xXFAyiEWhaFapQpikjLepWsIKpkn6imG0yhN+LLr79BzRkOpnYwRxtH9jue0cBWNGoltcnRYWndikpH0WjJlPuoL/zUFKfyGJQVVWHkRxPFoCRTK3aw44LrciO3ogBfaQUrll/klxg2R7PGAuOccRvlZ71QEAe/oDGv+r7tI+6tAX4ub8FcrGohIjFWt8KUGybR8sLAoYx2m/aioejFCOSnhd0qrWFKgebXfHlCUHdY5ad8kR/rCylXGjehUBXM+VwFIG2Yk4D5bZg9B8XF5L8RWuYnPZPaJONWL8yCFMxXoLKRXxk/ltb71Zs+e0ZOvw3lN0pKhibZaNuybRoQGiyeu4z3XUECiBfd1ncyilOhITlVFJ/x77ZtJfO+5Ql05B96ypfgnf2uqfwoywGyg19fvb169QG0hLyP/hqzN/P3/wioBFG5kiBY28Axi4vHLjr2IfnHnlW3GhbNPscIuNhKcGAKzEm7De4YZ8c6uCPsREBZ0wKE2eTt29YkcnsjqwpCgPLWocYn7isFgA2VJacsw9QLSM5udsxABXmqZs0g9nsN0KnuJJMflTYYapgXLtDYT681T9AcjyIj+7PISUZdlPOPlxxBgT3oIPWzw8AlqEKzxFOOhp9iUhe0QhMC1QDFtEswG8/CZ8LLzdDcFkb0a2kKwmPNsy8vLhJuy1ZRQslTsDOEQfYVdIOj0KJFuRFQKw0UmJuhgsU8+9s3sIyOEpXocOS0yAX9KQonly2nYe6n5hkNdV/yaRNTnWqoOvETRS7tdlBzZAe2lD1Yoxdr7Fo+2AN7iBQ0bwFIiNHyI3i57RVwjz116SUIUVEau6of9GW6E42eEb2giPD7YWgQpxgacSdUjQSF+yONbDC44bjrA8pimQYEFyAbec0ymBF58Uk7JVfpqaMTe0I120d34Cl+W3d+qKMFFteil78PEG/g0YsHm2ZUpNxJNtUcONsca+/zP0MaNz/B4EhqOFsnzrPEZ9w3DiTWuM7jh8jEfv3xan7lN8u/40nkN47HoGXRD+RCzvcOuk6g8nyUteJGgwOYI1QW3IK3M8bPvLq4McCIGeuBCjRiK6kJUZNUao1cAI6H6ZQNVr2cY8MVGKyz2Ey16VUXkSHZX9h1CyysUth5M2D5stbQEGsAOSgl0V1CLZgAscfA0WbHxHrdyzUkIoCTrKsUt8NZOf+t4el/WtWAqXqjF5dLyg8UFlOEdEg1BIihz8imD82lFMJeOx7x3wwwNZQeNLbbxd9eUvzbvS+WqdKgPsyNR92XS69l7MUKJgFKX3i1iAcV6AWKBI2muqnb8lbniyV8o2CWm4BgAAIrBeADORf1fPHv35rlGSe/WFs6G9P8dN23AxwfjI0wpZpB4kR7Qiq6TjZV9ImrCtLwe34GcyPU047HZ5dx+gf4uo6+jhMORAingX5i3YtuA4hCzs0OzqP0OvoHBIdSpEDmzrMF4r6NfsSQQgPYhm2oEcUP8biSuVjkmY1EbsBz0rUCd+YFSMQA3DkQeJ4FKk+z4iOkw80wS5J9F62yfVZ/ntm/dimikEZU50+VSh7UIrqYhSzho1znmc8ZJ9Ao83lm88ga1sULz0YNEK2tdmG6uzSlth1RckFUATWK+GBW53/ngBEb+dFu65xjtwjoDWqiyymKHPb5igbB51nGvWUa7eDZBtwY9kUTOAmkotY5mo9ikXDloOD4C5kDtsKCd/IKIf0XQvDXjiYliEeFE4LAtjX5vujNQAjbhI945vmQxePAjo6I0YxK6KqX8g+J66Z7dpT4NQHqlNnltuSqdSMgBqXLEbDDAj8ggKkUuR77DZ2WFhWBVdrh416Y6I/0TCCcE2ShBMB8yQ+lnUb2uMJma5AIOLG7X1mwxaUp8TXtkFbjfX58XaMpR5cxotA40d+vvEHohuW+T5SWVwKI2r6AsPBK4RUChF+tMNroFNIasEqtdpEPniQoCRQLdis2ClIq9AEwmEHnTjjbAlP6Ap9jENNgCtwr4p4BFHs/f3qIrY4OV2g2XkQQcsA0linJflUQRQeQhbsupR2C8cwGLH5J1Jm+8HUD2A+gOkb+LFy3sIWVEB9P8s+LnSNzZ/4r8bpm/Go+fz8HlHmavIzQl3v4eBav/lfG/kBa7Sig7pXZMP8+k37w7Ou16imd0bNypT7mlpGe+70sizvnMTI8XE6bsb0pKbdxgJxFJTBcbbEKnsypRfa1vZQ9FVwY6HhPDyq5tdwTu9wfsgj1ZpngOaMBbEKu2XPcBhZ4jvni8U5ujYunUHpvM7dmcWtJyq0l8X6dr3W2qCUHBeyodiWhRB3WocPykxyVnMNac1hfkkelYvng9HBUwwvvK+Yys7mxpxahdJ+yLqfZGNlE6AJ6eNf402j3ol2tQDudf7qxiZTB38c8wprxBs34SLoH5zioCchHdH5YjCLXDCUupfuf+kP2ThDcmubS9n4bF3d2JB06vBxGzhl5EMJ7Zzm+BuyHxy5bjm8Hj6aQH5ch1Q/8uZ986OZlMnLsftLY28tkzFLDlHF47A89AoY/fSPyCllP7t2VH7h2QRZ0/qIQ8XljHQQFzWBYBcvHs0qGhw2aSZC74N6xNi2bynoYaDfQCnxfOwX2564QTEYBbs899cLlEUnvn+Twnom0BV/Km/bgvWWGHY00iHL4RHj4lL+CK8ENkB5bHJ+qjp7MFE5OnriShKXPfsZJsDxp7vSFi9kAZvtM2yRf2TLJug3WslNl0V2585AojnUuysVITroe2HwsMWGsjMuzxbjyLffs0X/EY0+O1H7WeSQsLQxV+AS3jEKZsa9D2OftmDitFtn55dLLsHPJ3YkdsqakVEZY7Biz1OigloeXSy1FX24iv+/5ZQbGvYgPMTTseCLK3rXj+HJPGStl6MkOaivEn5OAuUenMVC5jcn0/pCUflzwnBCDzY3s42IMpvl++IkM5suz/Zxx9/P7Qxr/f7vTQrs3RYjHW8cP+fzq5et/YVEnf+Xeb4/4YDhbn0gjxzxUA4m5y4I//VvvMc+TH0vZGXZFf9DiwGygj0zsw5gqHYTurpMRjMVpUeDlsCjOOD63IJnH7kV2eXFxsXRUPLwcK/szBT3sHIsz2Us6ObaKt7nnse5vQg9HhSOy9g/kocTLPd3UPXA8A5O2FngGRkDiAr1wvyHmIdn8oxRCH+oRBsIVJSkBjvA3kMGUuQPA+/ADET0dHaDXyQPDbxFz++7psihQ030yzV/+sDfuvbDW9bvgoSTCfa+MLFqIFrosJSRCGH36/kI3i8d3F6dBuLuEmDy4xFj/PXeJOX5jOeW4A8Q/MYFgPnniJpFfJNLHcj6SxwZxPHqbocHJfwE/b+H1'))
assert hashlib.sha256(snapshot_source).hexdigest() == '9322be003ffb1fe23c1a20910e8f5bc559e985b1635d9e7bd7e67efdecbbc3ee'
(PROJECT / 'src/snapshot_rag.py').write_bytes(snapshot_source)
import snapshot_rag
snapshot_rag = importlib.reload(snapshot_rag)
SNAPSHOT_FOLDER = PROJECT / 'fever_snapshot'
SNAPSHOT_FOLDER.mkdir(exist_ok=True)
snapshot_archive = DRIVE_PROJECT / 'FEVER_Wikipedia_Snapshot.zip'
if not snapshot_archive.is_file(): raise FileNotFoundError('FEVER_Wikipedia_Snapshot.zip must be in FEVER_Project')
with zipfile.ZipFile(snapshot_archive) as archive:
    if set(archive.namelist()) != {'snapshot.sqlite','snapshot_manifest.json'}: raise ValueError('Unexpected snapshot ZIP contents')
    manifest_bytes = archive.read('snapshot_manifest.json')
    if hashlib.sha256(manifest_bytes).hexdigest() != '1f94438dfac69b7390a97256722c562480c122f2a387da517d2d1cfaae7b1ded': raise ValueError('Snapshot manifest differs from the verified export')
    for name in ['snapshot.sqlite','snapshot_manifest.json']:
        content = archive.read(name)
        target = SNAPSHOT_FOLDER / name
        if target.exists() and target.read_bytes() != content: raise ValueError('Existing source snapshot differs')
        if not target.exists(): target.write_bytes(content)
probe = snapshot_rag.SnapshotCorpus(SNAPSHOT_FOLDER, fever.sha_file(PROJECT / 'data/eval_inputs.jsonl'))
try: print('Snapshot verified:',probe.manifest['packaged_pages'],'selected full records; zero Wikipedia downloads per claim.')
finally: probe.close()


Snapshot verified: 24319 selected full records; zero Wikipedia downloads per claim.


## 4. Recheck validation selection and set evaluation output
Selection reads only the training runs' validation histories. The expected file
hashes were fixed when we selected the model, before this full evaluation.

In [6]:
EXPECTED_STATE_HASHES = {'1': '982723bafdf2a3e631321606bc427b4c87abc6e2d1f386f45c27866efb9def2a', '2': 'bfe5f5354747fd25a5e54100cd06f17aed6a08104b4d8a9d7313d2204716fc67', '3': 'e2fe1583b3c925e84871629123096f63f20a3600ac6b92867a9ac45d697590cc'}

TRAINING_ROOT = DRIVE_PROJECT / "runs/roberta_full_unfreeze_v1/training"
SELECTION = paired.select_validation(TRAINING_ROOT)
for candidate in SELECTION["candidates"]:
    if candidate["state_sha256"] != EXPECTED_STATE_HASHES[str(candidate["last_blocks"])]:
        raise ValueError("Training history changed since model selection; inspect before evaluating")
assert SELECTION["winner"]["last_blocks"] == 3
import pandas as pd
display(pd.DataFrame([{"Unfrozen blocks": r["last_blocks"],
    "Validation macro-F1 (%)": 100*r["validation_macro_f1"],
    "Validation loss": r["validation_loss"], "Selected step": r["selected_step"]}
    for r in SELECTION["candidates"]]).round(4))
EVALUATION_NAME = "roberta_paired_snapshot_dev_v1" # @param {type:"string"}
SESSION_MINUTES = 180 # @param {type:"integer"}
RUN_PAIRED_EVALUATION = True # @param {type:"boolean"}
if not 1 <= SESSION_MINUTES <= 300: raise ValueError("Use 1–300 minutes")
EVALUATION_OUT = DRIVE_PROJECT / "runs" / EVALUATION_NAME
EVALUATION_OUT.mkdir(parents=True, exist_ok=True)
print("Selected model: last 3 blocks; validation macro-F1:", round(100*SELECTION["winner"]["validation_macro_f1"],2), "%")
print("Target: all", len(EVAL_ROWS), "claims, TWO models, same retained RAG input.")
print("Results:", EVALUATION_OUT)
EMBEDDING = fever.remote_metadata("sentence-transformers/all-MiniLM-L6-v2",
    requested_revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41")
print("Only this ~0.08 GiB embedding model may download; both RoBERTa checkpoints already exist.")

,Unfrozen blocks,Validation macro-F1 (%),Validation loss,Selected step
0,1,66.2971,0.6809,11000
1,2,68.2425,0.6491,11000
2,3,69.4235,0.6346,11000


Selected model: last 3 blocks; validation macro-F1: 69.42 %
Target: all 19998 claims, TWO models, same retained RAG input.
Results: /content/drive/MyDrive/FEVER_Project/runs/roberta_paired_snapshot_dev_v1
sentence-transformers/all-MiniLM-L6-v2 revision 1110a243fdf4706b3f48f1d95db1a4f5529b4d41 weight download approximately 0.08 GiB before 4-bit runtime quantization
Only this ~0.08 GiB embedding model may download; both RoBERTa checkpoints already exist.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 5. Run / resume snapshot RAG evaluation — enabled
Reads the selected full FEVER article records locally, builds/deletes a temporary
RAG database per claim and evaluates both unchanged models on identical retained
input. No live article fetching or LLM generation occurs. Progress, source reading,
embedding, predictions and rough ETA are logged. The first 10 paired outputs form
an execution check, then the same experiment continues toward all 19,998.
Three consecutive preparation failures stop for review; failures are not relabeled NEI.
Initial model hashing/loading can take time; later sessions skip completed pairs.


In [ ]:
CONDITIONS = ["roberta_original_rag512", "roberta_last3_rag512"]
if RUN_PAIRED_EVALUATION:
    import time
    started = time.monotonic()
    corpus = snapshot_rag.SnapshotCorpus(SNAPSHOT_FOLDER, fever.sha_file(PROJECT / "data/eval_inputs.jsonl"))
    try:
        counts = [sum(p.name != "protocol.json" for p in (EVALUATION_OUT / "predictions" / name).glob("*.json")) for name in CONDITIONS]
        if min(counts) < 10:
            print("Execution check: prepare snapshot RAG and evaluate the first 10 pairs; target remains all 19,998.", flush=True)
            paired.run_pairs(EVAL_ROWS, PROJECT, EVALUATION_OUT, SELECTION, EMBEDDING,
                minutes=SESSION_MINUTES, max_new_claims=10-min(counts), corpus=corpus)
            print("Execution check saved. These are not final accuracy results.", flush=True)
        remaining = SESSION_MINUTES - (time.monotonic()-started)/60
        if remaining >= 1:
            CONDITIONS = paired.run_pairs(EVAL_ROWS, PROJECT, EVALUATION_OUT, SELECTION,
                EMBEDDING, minutes=remaining, corpus=corpus)
        else:
            print("Session budget reached. Run all again to resume.")
    finally:
        corpus.close()
else:
    print("Evaluation disabled.")


Execution check: prepare snapshot RAG and evaluate the first 10 pairs; target remains all 19,998.


paired_roberta:   0%|          | 0/19998 [00:00<?, ?claim/s]

[18:32:09 UTC] paired_roberta | session_started {"log_path": "/content/drive/MyDrive/FEVER_Project/runs/roberta_paired_snapshot_dev_v1/monitor/paired_roberta/session_1793989929095095756.jsonl"}
[18:32:09 UTC] paired_roberta | loading_embedding_model {"model": "sentence-transformers/all-MiniLM-L6-v2", "device": "cuda", "dtype": "float32"}


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (10 outputs left out)

Token indices sequence length is longer than the specified maximum sequence length for this model (1144 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1144 > 512). Running this sequence through the model will result in indexing errors


[18:32:52 UTC] paired_roberta | claim #140846 | embedding_batch {"embedded": 10, "total_texts": 10}
[18:32:52 UTC] paired_roberta | claim #140846 | evidence_saved {"evidence_status": "READY", "supplied_passages": 39, "preparation_seconds": 0.169, "wikipedia_http_requests": 0}
[18:32:52 UTC] paired_roberta | claim #140846 | roberta_inference {"input_tokens": 512, "untruncated_tokens": 1144, "truncated": true}
[18:32:52 UTC] paired_roberta | claim #140846 | roberta_inference {"input_tokens": 512, "untruncated_tokens": 1144, "truncated": true}
[18:32:52 UTC] paired_roberta | claim #140846 | pair_predictions {"claim_id": 140846, "original": "SUPPORTS", "selected": "SUPPORTS", "identical_context": true, "identical_retained_input": true}
[18:32:52 UTC] paired_roberta | claim #140846 | claim_saved {"prediction": "PAIR_COMPLETE", "completed": 11, "total": 19998, "invalid_total": 0, "claim_seconds": 0.48, "recent_mean_seconds": 0.48, "rough_remaining_hours": null, "input_tokens": null, "output_

Token indices sequence length is longer than the specified maximum sequence length for this model (655 > 512). Running this sequence through the model will result in indexing errors


Streaming output truncated to the last 5000 lines.
[19:10:26 UTC] paired_roberta | claim #144161 | roberta_inference {"input_tokens": 512, "untruncated_tokens": 964, "truncated": true}
[19:10:26 UTC] paired_roberta | claim #144161 | pair_predictions {"claim_id": 144161, "original": "SUPPORTS", "selected": "SUPPORTS", "identical_context": true, "identical_retained_input": true}
[19:10:27 UTC] paired_roberta | claim #144161 | claim_saved {"prediction": "PAIR_COMPLETE", "completed": 2584, "total": 19998, "invalid_total": 0, "claim_seconds": 0.99, "recent_mean_seconds": 1.09, "rough_remaining_hours": 5.29, "input_tokens": null, "output_tokens": null, "prediction_path": ""}
[19:10:27 UTC] paired_roberta | claim #205726 | claim_started {"claim_id": 205726}
[19:10:27 UTC] paired_roberta | Claim: First Motion Picture Unit produced zero informative films.
[19:10:27 UTC] paired_roberta | claim #205726 | reading_snapshot_article {"page_id": "First_Motion_Picture_Unit", "rank": 1}
[19:10:27 UTC] p

## 6. Progress and final comparison
Final metrics appear only when both systems cover all 19,998 IDs. INVALID stays
in the denominator. Metrics are label accuracy, macro-F1, per-class precision,
recall and F1, confusion matrices, and paired counts of fixes versus new errors.
Snapshot record/character-span citations are retained; this reports label metrics, not an official strict FEVER evidence score.

In [ ]:
for name in CONDITIONS:
    folder = EVALUATION_OUT / "predictions" / name
    count = sum(p.name != "protocol.json" for p in folder.glob("*.json"))
    print(name, count, "/", len(EVAL_ROWS))
METRICS = paired.report(PROJECT, EVALUATION_OUT, CONDITIONS)
if len(METRICS) == 2:
    display(pd.DataFrame([{"System": name, "Claims": m["n"],
        "Accuracy (%)": 100*m["accuracy"], "Macro-F1 (%)": 100*m["macro_f1"],
        "Invalid": m["invalid"]} for name,m in METRICS.items()]).round(2))
    for name,m in METRICS.items():
        print(name)
        display(pd.DataFrame([{"Class": label, **{k+" (%)": 100*v for k,v in values.items()}}
            for label,values in m["per_class"].items()]).round(2))
        display(pd.DataFrame(m["confusion_matrix"], index=fever.LABELS,
            columns=fever.LABELS+["INVALID"]))
fever.monitor_status(EVALUATION_OUT, "paired_roberta", last_events=5)

## 7. Inspect a saved pair
An empty ID selects the most recently written prediction. Reference labels are
shown only after the model prediction has been saved. These labels never enter
either model's input. RoBERTa has probabilities and retained tokenized input,
rather than a generated textual explanation.

In [ ]:
INSPECT_CLAIM_ID = "" # @param {type:"string"}
for name in CONDITIONS:
    print("\n", name)
    fever.inspect_saved(EVALUATION_OUT, name,
        claim_id=int(INSPECT_CLAIM_ID) if INSPECT_CLAIM_ID.strip() else None,
        show_reference=True, root=PROJECT, show_full=True)